In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rk4(params, f, yo, t0, tf, h, B, phi2):
    C2 = params["C2"]
    C3 = params["C3"]
    C4 = params["C4"]
    
    A21 = params["A21"]
    A31 = params["A31"] 
    A32 = params["A32"]
    A41 = params["A41"]
    A42 = params["A42"]
    A43 = params["A43"]
    
    B1 = params["B1"]
    B2 = params["B2"]
    B3 = params["B3"]
    B4 = params["B4"]
    
    N = int((tf - t0) / h)
    
    Y = [yo]
    T = [t0]
    
    for i in range(N):
        t_n = T[-1]
        y_n = Y[-1]
        
        k1 = f(t_n, y_n, B, phi2)
        k2 = f(t_n + h*C2, y_n + h*A21*k1, B, phi2)
        k3 = f(t_n + h*C3, y_n + h*(A31*k1 + A32*k2), B, phi2)
        k4 = f(t_n + h*C4, y_n + h*(A41*k1 + A42*k2 + A43*k3), B, phi2)
        
        y_next = y_n + h*(B1*k1 + B2*k2 + B3*k3 + B4*k4)
        t_next = t_n + h
        
        Y.append(y_next)
        T.append(t_next)
    
    return np.array(T), np.array(Y)

def xdot(t, x, y, z, sigma):
    return sigma * (y - x)

def ydot(t, x, y, z, rho):    
    return x * (rho - z) - y

def zdot(t, x, y, z, beta):
    return x * y - beta * z

def sistema(t, Y, sigma, rho, beta):
    x, y, z = Y
    return np.array([xdot(t, x, y, z, sigma),
        ydot(t, x, y, z, rho),
        zdot(t, x, y, z, beta)])

def multipaso(f, y0, t0, tf, h, params_, sigma, rho, beta):
    N = int((tf - t0) / h)
    tiempos = [t0]
    valores = [y0]
    t = t0
    y = y0
    
    def f_(t, Y):
        return sistema(t, Y, sigma, rho, beta)
    
    T_rk, Y_rk = rk4(params_, f_, y0, t0, t0 + 2*h, h, None, None)
    
    tiempos.append(T_rk[1])
    valores.append(Y_rk[1])
    tiempos.append(T_rk[2])
    valores.append(Y_rk[2])

    for i in range(2, N):
        t_n = tiempos[-i]
        y_n = valores[-i]
        t_n1 = tiempos[-(i+1)]
        y_n1 = valores[-(i+1)]
        t_n2 = tiempos[-(i+2)]
        y_n2 = valores[-(i+2)]
        
        f_n = sistema(t_n, y_n, sigma, rho, beta)
        f_n1 = sistema(t_n1, y_n1, sigma, rho, beta)
        f_n2 = sistema(t_n2, y_n2, sigma, rho, beta)
        
        y_sig = y_n + (h/12) * (23 * f_n - 16 * f_n1 + 5 * f_n2)
        t_sig = t_n + h
        
        tiempos.append(t_sig)
        valores.append(y_sig)
    
    return np.array(tiempos), np.array(valores)

params_ = {"C2": 0.5, "C3": 0.5, "C4": 1.0, "A21": 0.5, "A31": 0, "A32": 0.5, "A41": 0, "A42": 0, "A43": 1.0, "B1": 1/6, "B2": 1/3, "B3": 1/3, "B4": 1/6}

yo = np.array([0.2, 1.5, 0.1])
t0 = 0.0
tf = 100
h = 0.01
sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0